# Datamine XVALID Process Example

This notebook demonstrates how to configure and run the **`xvalid`** process wrapper in `dmstudio`.

## Process Description

## XVALID

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

XVALID assists in the selection of parameters for grade estimation, using the cross-validation method.

The input data file is the sample data which will later be used for estimating grades into a block model. For kriging it allows different model variograms to be tested and compared. For inverse power of distance it allows different powers to be compared.

The input to **XVALID** is consistent with the input required for the grade estimation process [ESTIMA](<estima.md>). In fact the three input parameter files are identical for the two processes.

The cross-validation method works by removing each point in turn from the data file and estimating its value from the remaining data. In this way a table of actual and estimated values is created. A detailed statistical analysis is then carried out comparing the actuals and estimates. One or more of the estimation parameters can then be changed and the process rerun to see whether the new parameters improve the results of the statistical analysis. The method is therefore iterative, requiring several runs to establish the best set of parameters.

**XVALID** uses the block model grade estimation process [ESTIMA](<estima.md>) to do the actual interpolation. Therefore part of the text on the Output window is the same as for [ESTIMA](<estima.md>), and will refer to a block model rather than points. However the number of discretisation points is set to 1x1x1 so in effect point estimation is being used.

There are three input parameter files all of which can be set up using the [ESTIMATE](<estimate.md>) process or the Table Editor:

* **SRCPARM** : Search Volume Parameter File

* **ESTPARM** : Estimation Parameter File

* **VMODPARM** : Variogram Model Parameter File

and two other input files:

* **IN** : Sample File

* **VGRAM** : Experimental Variogram File

The results of the cross-validation are written to three separate output files, all of which are optional:

* **XVSAMPS** : cross-validated output sample file

* **XVSTATS** : cross-validation statistics

* **SAMPOUT** : sample output file containing weights for each estimate

The statistics are also displayed in the **Output** window as illustrated in the example. When the first set of results have been calculated and displayed you are then able to select from a menu which allows you to edit one or more of the parameter files or examine the results, and then rerun the cross-validation:

1 - edit search volume parameter file spar10

2 - edit estimation parameter file epar10

3 - edit variogram model parameter file vpar10

4 - examine variogram model interactively (**[VARFIT](<varfit.md>)**)

5 - examine cross-validation statistics file xxstats

6 - delete cross-validation statistics file xxstats

7 - create plot of actual v estimate

8 - re-run cross-validation

0 - exit cross-validation

Option 4 allows you to use the interactive variogram fitting process [VARFIT](<varfit.md>). In order to use this option you must have specified the experimental variogram file **VGRAM** when selecting **XVALID**.

Options 5 and 6 are only available if the output file **XVSTATS** has been specified.

Option 7 is not available if multiple fields are being cross-validated ie if there is more than one record in the estimation parameter file.

### Input Files:

* **in** (Undefined):
  Input sample data to be cross validated. This must contain X,Y and Z fields and at
  least one grade field.
  Required=Yes

* **srcparm**:
  Search volume parameter file. This contains 24 compulsory fields defining the search
  volume and the number of samples needed for grade estimation. More than one search
  volume may be defined. All fields are numeric:
  Required=No

* **estparm**:
  Estimation parameter file. Each record in the file describes an estimation method and
  its associated parameters. The fields are dependent on the estimation methods selected.
  General fields:
  Required=No

* **vmodparm**:
  Variogram model parameter file. Each record in this file defines a variogram model type
  and its parameters. Only the VREFNUM field is compulsory.
  Required=No

* **vgram** (Variogram - Experimental):
  Experimental variogram file, as created by the variogram calculation process
  [VGRAM](<vgram.md>). This experimental variogram file will have been used by the
  variogram fitting process [VARFIT](<varfit.md>) in order to derive the variogram model
  defined by **VMODPARM** . This is only required if you want to use access the variogram
  display and fitting process **VARFIT** from within **XVALID**.
  Required=No

### Output Files:

* **xvsamps** (Undefined):
  Cross-validated output sample file. This contains all the fields from the IN sample
  data file, plus each grade estimate and associated secondary fields such as kriged
  variance.
  Required=No

* **xvstats** (Undefined):
  Output file containing a summary of the input parameters and the cross-validation
  statistics. It includes a single record for each estimate. The 23 fields in the file are
  summarised below. If the file already contains all 23 fields then additional records are
  appended to the file. If the file does not contain all 23 fields, or if the file does
  not exist, then a new file is created.
  Required=No

* **sampout** (Undefined):
  Output sample file containing details of weights for each sample for each estimate.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate of sample data in **IN** file. If not specified, then X is assumed.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate of sample data in **IN** file. If not specified, then Y is assumed.
  Default=Undefined
  Required=Yes

* **z** (Numeric : IN):
  Z coordinate of sample data in **IN** file. If not specified, then Z is assumed.
  Default=Undefined
  Required=Yes

* **zone1_f** (Any : IN, ESTPARM):
  First field for zonal control. The field must exist in the **IN** file and in the
  **ESTPARM** file.
  Default=Undefined
  Required=No

* **zone2_f** (Any : IN, ESTPARM):
  Second field for zonal control. The field must exist in the IN file and in the
  **ESTPARM** file.
  Default=Undefined
  Required=No

* **key** (Numeric : IN):
  Key field used to specify the field limiting the number of samples for estimation. The
  field must exist in the **IN** file.
  Default=Undefined
  Required=No

* **length_f** (Numeric : IN):
  Field used for length weighting in **IPD**. The field must exist in the IN file.
  Default=Undefined
  Required=No

* **dens_f** (Numeric : IN):
  Field used for density weighting in **IPD**. The field must exist in the **IN** file.
  Default=Undefined
  Required=No

### Parameters:

* **sminfac**:
  Multiplying factor which is applied to the first search volume, and used to calculate
  the exclusion volume for estimation. Samples lying within the exclusion volume are not
  used for the estimation. The factor must be greater than 0 and less than 1. The
  exclusion volume is concentric with the search volume.
  Range=0,1
  Values=Undefined
  Default=0.0001
  Required=No

* **print**:
  Display control: Options: 0: Minimum output.; 1: Maximum output.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('xvalid')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute xvalid
print("Running xvalid...")
dm_cmd.xvalid(
    in_i='t_assays',  # required
    sampout_o='t_xvalid_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    # srcparm_i='t_spar',  # optional
    # estparm_i='t_epar',  # optional
    # vmodparm_i='t_vpar',  # optional
    # vgram_i='optional',  # optional
    # xvsamps_o=['t_xvalid_out'],  # optional
    # xvstats_o=['t_xvalid_out'],  # optional
    # zone1_f_f='optional',  # optional
    # zone2_f_f='optional',  # optional
    # key_f=['BHID'],  # optional
    # length_f_f='optional',  # optional
    # dens_f_f='optional',  # optional
    # sminfac_p=0.0001,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("xvalid execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_xvalid_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")